# Notebook 7: Logistic Regression
**DCS 404 — Data Science and Machine Learning**

---

## Learning Objectives
- Explain why linear regression fails for classification
- Derive the logistic (sigmoid) function
- Understand maximum likelihood estimation
- Implement logistic regression for binary and multiclass problems
- Evaluate classifiers with the right metrics

## 1. Why Not Linear Regression for Classification?

Suppose we want to predict whether an email is spam (1) or not (0).

Linear regression can predict values outside [0, 1] — e.g., 1.7 or −0.3 — which make no sense as probabilities.

We need a model that outputs a value **between 0 and 1**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_curve, roc_auc_score
)
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)

# 1D example: study hours → pass/fail
hours = np.array([0.5,1,1.5,2,2.5,3,3.5,4,4.5,5,5.5,6,6.5,7,8,9,10])
passed = np.array([0,0,0,0,0,0,1,0,1,1,1,1,1,1,1,1,1])

lin_reg = LinearRegression().fit(hours.reshape(-1,1), passed)
log_reg = LogisticRegression().fit(hours.reshape(-1,1), passed)

x_plot = np.linspace(0, 10, 200)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, model, title, extra in zip(
        axes,
        [lin_reg, log_reg],
        ['Linear Regression (bad for classification)', 'Logistic Regression (correct)'],
        [True, False]):
    ax.scatter(hours, passed, color='steelblue', zorder=5)
    preds = model.predict(x_plot.reshape(-1,1))
    ax.plot(x_plot, preds, 'r-', linewidth=2)
    if extra:
        ax.axhline(0, color='gray', linestyle='--')
        ax.axhline(1, color='gray', linestyle='--')
    ax.set_xlabel('Study hours')
    ax.set_ylabel('P(pass)')
    ax.set_title(title)
plt.tight_layout()
plt.show()

## 2. The Sigmoid (Logistic) Function

We model the probability of class 1 as:

$$P(y=1 | x) = \sigma(z) = \frac{1}{1 + e^{-z}}, \quad z = w_0 + w_1 x_1 + \cdots + w_p x_p$$

Properties of $\sigma$:
- Output is always in (0, 1)
- $\sigma(0) = 0.5$ — the **decision boundary**
- Smooth and differentiable (needed for gradient descent)

In [ ]:
z = np.linspace(-8, 8, 300)
sigmoid = 1 / (1 + np.exp(-z))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(z, sigmoid, 'steelblue', linewidth=2)
ax.axhline(0.5, color='red', linestyle='--', label='Decision boundary (p=0.5)')
ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
ax.fill_between(z, 0.5, sigmoid, where=sigmoid>0.5, alpha=0.1, color='green', label='Predict class 1')
ax.fill_between(z, sigmoid, 0.5, where=sigmoid<0.5, alpha=0.1, color='red', label='Predict class 0')
ax.set_xlabel('z (linear combination of features)')
ax.set_ylabel('σ(z) = P(y=1)')
ax.set_title('The Sigmoid Function')
ax.legend()
plt.tight_layout()
plt.show()

## 3. The Loss Function: Binary Cross-Entropy

We maximise the likelihood (or equivalently minimise the **binary cross-entropy loss**):

$$\mathcal{L} = -\frac{1}{n} \sum_{i=1}^n \left[ y_i \log(\hat{p}_i) + (1-y_i) \log(1-\hat{p}_i) \right]$$

Intuition:
- When $y_i = 1$ and $\hat{p}_i \to 0$: $-\log(0) = \infty$ — large penalty
- When $y_i = 1$ and $\hat{p}_i \to 1$: $-\log(1) = 0$ — no penalty

In [ ]:
# Visualise the loss for y=1
p = np.linspace(0.001, 0.999, 300)
loss_y1 = -np.log(p)
loss_y0 = -np.log(1 - p)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(p, loss_y1, label='Loss when y=1 (penalise low predictions)', color='green')
ax.plot(p, loss_y0, label='Loss when y=0 (penalise high predictions)', color='red')
ax.set_xlabel('Predicted probability p̂')
ax.set_ylabel('Loss')
ax.set_title('Binary Cross-Entropy Loss')
ax.legend()
ax.set_ylim(0, 5)
plt.tight_layout()
plt.show()

## 4. Binary Classification: Breast Cancer Dataset

In [ ]:
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target  # 0=malignant, 1=benign

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, C=1.0))
])
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
y_prob = pipe.predict_proba(X_test)[:, 1]

print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=cancer.target_names))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks([0,1]); axes[0].set_xticklabels(cancer.target_names)
axes[0].set_yticks([0,1]); axes[0].set_yticklabels(cancer.target_names)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix')
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm[i,j]), ha='center', va='center',
                     color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=14)

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)
axes[1].plot(fpr, tpr, linewidth=2, label=f'AUC = {auc:.3f}')
axes[1].plot([0,1],[0,1], 'k--', alpha=0.5, label='Random classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()
plt.tight_layout()
plt.show()

## 5. Multiclass Classification (One-vs-Rest)

Logistic regression extends to K classes using either:
- **One-vs-Rest (OvR)**: K binary classifiers, one per class
- **Softmax (multinomial)**: single model with softmax output

In [ ]:
iris = load_iris()
X_i, y_i = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X_i, y_i, test_size=0.2, random_state=42)

for solver_name, multi in [('OvR', 'ovr'), ('Softmax', 'multinomial')]:
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(multi_class=multi, solver='lbfgs', max_iter=200))
    ])
    pipe.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, pipe.predict(X_te))
    print(f'{solver_name:10s}: accuracy = {acc:.3f}')

In [ ]:
# Decision boundary visualisation (2 features only)
from matplotlib.colors import ListedColormap

X2 = iris.data[:, :2]  # sepal length and width only
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X2, y_i, test_size=0.2, random_state=42)

pipe2 = Pipeline([('scaler', StandardScaler()),
                  ('clf', LogisticRegression(multi_class='ovr', max_iter=200))])
pipe2.fit(X_tr2, y_tr2)

x_min, x_max = X2[:,0].min()-0.5, X2[:,0].max()+0.5
y_min, y_max = X2[:,1].min()-0.5, X2[:,1].max()+0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                     np.linspace(y_min, y_max, 200))
Z = pipe2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(['#FF9999','#99FF99','#9999FF']))
scatter = ax.scatter(X2[:,0], X2[:,1], c=y_i, cmap=ListedColormap(['#CC0000','#00AA00','#0000CC']),
                     edgecolors='white', s=40)
ax.set_xlabel(iris.feature_names[0])
ax.set_ylabel(iris.feature_names[1])
ax.set_title('Logistic Regression Decision Boundaries (Iris)')
plt.colorbar(scatter, ax=ax, ticks=[0,1,2])
plt.tight_layout()
plt.show()

## Exercises

1. Using the breast cancer dataset, tune the regularization parameter `C` (try `C = [0.001, 0.01, 0.1, 1, 10, 100]`) with 5-fold cross-validation. Plot validation accuracy vs `C`.
2. Compute **precision, recall, and F1-score** manually from the confusion matrix values (TP, FP, FN, TN). Verify against `classification_report`.
3. The **ROC AUC** measures a classifier's ability regardless of the decision threshold. Change the threshold from 0.5 to 0.3 for the cancer classifier. How does the confusion matrix change? What trade-off are you making?
4. Apply logistic regression to the **digits dataset** (`sklearn.datasets.load_digits`) — a 10-class problem. What accuracy do you achieve?

## Reflection Questions
- What does the coefficient for a feature tell you in logistic regression? (Think in terms of odds ratios.)
- Precision and Recall are often in tension. In a cancer screening test, which would you prioritise and why?
- What is the difference between `predict()` and `predict_proba()` — when would you use each?

---
**Next →** `08_svm.ipynb`